# Tarea 6 y 7 - Procesamiento y clasificación de audio

En esta actividad uso el conjunto **Free Spoken Digit Dataset (FSDD)**. Son grabaciones de los dígitos del 0 al 9 hechas por varias personas. Primero reviso las señales y sus espectrogramas; después obtengo características de audio para clasificar el dígito y también al hablante.

A diferencia de la referencia, aquí uso espectrogramas Mel, características MFCC y espectrales, KNN y regresión logística. La comparación de señales se hace con DTW sobre MFCC.

## 1. Librerías y datos

La siguiente celda se ejecuta una sola vez si faltan librerías en el ambiente local.

In [ ]:
%pip install librosa soundfile numpy pandas matplotlib scikit-learn

In [ ]:
from pathlib import Path

carpeta_datos = Path('fsdd')

# descargo las grabaciones solamente si todavía no están en esta carpeta
if not carpeta_datos.exists():
    !git clone --depth 1 https://github.com/Jakobovski/free-spoken-digit-dataset.git fsdd

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

semilla = 42
frecuencia_muestreo = 8000
np.random.seed(semilla)

In [ ]:
archivos = sorted(Path('fsdd/recordings').glob('*.wav'))

digitos = []
hablantes = []
for archivo in archivos:
    partes = archivo.stem.split('_')
    digitos.append(partes[0])
    hablantes.append(partes[1])

datos = pd.DataFrame({
    'archivo': [str(archivo) for archivo in archivos],
    'digito': digitos,
    'hablante': hablantes
})

print('Audios encontrados:', len(datos))
display(datos.head())
display(datos['digito'].value_counts().sort_index())
display(datos['hablante'].value_counts())

## 2. Revisión de las señales

Tomo cuatro dígitos de la misma persona para que las diferencias se vean principalmente por el sonido del dígito y no por cambiar de hablante.

In [ ]:
hablante_ejemplo = datos['hablante'].iloc[0]
digitos_ejemplo = ['0', '3', '6', '9']

figura, ejes = plt.subplots(2, 2, figsize=(12, 6))
for eje, digito in zip(ejes.ravel(), digitos_ejemplo):
    fila = datos[(datos['digito'] == digito) & (datos['hablante'] == hablante_ejemplo)].iloc[0]
    audio, sr = librosa.load(fila['archivo'], sr=frecuencia_muestreo)
    tiempo = np.arange(len(audio)) / sr
    eje.plot(tiempo, audio, linewidth=.8)
    eje.set_title(f'Dígito {digito} - {hablante_ejemplo}')
    eje.set_xlabel('tiempo (s)')
    eje.set_ylabel('amplitud')
plt.tight_layout()
plt.show()

In [ ]:
duraciones = []
for archivo in datos['archivo']:
    audio, sr = librosa.load(archivo, sr=frecuencia_muestreo)
    duraciones.append(len(audio) / sr)

datos['duracion_segundos'] = duraciones

plt.figure(figsize=(9, 4))
plt.boxplot([datos[datos['digito'] == str(numero)]['duracion_segundos'] for numero in range(10)], tick_labels=range(10))
plt.title('Duración de los audios por dígito')
plt.xlabel('dígito')
plt.ylabel('segundos')
plt.show()

datos.groupby('digito')['duracion_segundos'].mean().round(3)

In [ ]:
figura, ejes = plt.subplots(2, 2, figsize=(12, 8))
for eje, digito in zip(ejes.ravel(), digitos_ejemplo):
    fila = datos[(datos['digito'] == digito) & (datos['hablante'] == hablante_ejemplo)].iloc[0]
    audio, sr = librosa.load(fila['archivo'], sr=frecuencia_muestreo)
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=64, hop_length=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(mel_db, sr=sr, hop_length=128, x_axis='time', y_axis='mel', ax=eje)
    eje.set_title(f'Espectrograma Mel: dígito {digito}')
plt.tight_layout()
plt.show()

## 3. Características para cada grabación

Para cada audio guardo la media y desviación estándar de 20 MFCC, además de energía RMS, centroide espectral, rolloff, tasa de cruces por cero y duración.

In [ ]:
caracteristicas = []

for archivo in datos['archivo']:
    audio, sr = librosa.load(archivo, sr=frecuencia_muestreo)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20, n_fft=512, hop_length=128)
    media_mfcc = mfcc.mean(axis=1)
    desviacion_mfcc = mfcc.std(axis=1)
    rms = librosa.feature.rms(y=audio, frame_length=512, hop_length=128).mean()
    centroide = librosa.feature.spectral_centroid(y=audio, sr=sr, n_fft=512, hop_length=128).mean()
    rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr, roll_percent=.85, n_fft=512, hop_length=128).mean()
    cruces_cero = librosa.feature.zero_crossing_rate(audio, frame_length=512, hop_length=128).mean()
    duracion = len(audio) / sr
    caracteristicas.append(np.concatenate([media_mfcc, desviacion_mfcc, [rms, centroide, rolloff, cruces_cero, duracion]]))

matriz_caracteristicas = np.array(caracteristicas)
print('Tamaño de la matriz:', matriz_caracteristicas.shape)
print('40 valores de MFCC + 5 características de audio = 45 características')

## 4. Clasificación del dígito

Separo 20% para prueba. El escalamiento se ajusta solamente con los datos de entrenamiento. Comparo KNN, que decide por audios parecidos, contra regresión logística.

In [ ]:
caracteristicas_entreno, caracteristicas_prueba, digito_entreno, digito_prueba = train_test_split(
    matriz_caracteristicas, datos['digito'], test_size=.20, random_state=semilla, stratify=datos['digito']
)

escalador = StandardScaler()
caracteristicas_entreno = escalador.fit_transform(caracteristicas_entreno)
caracteristicas_prueba = escalador.transform(caracteristicas_prueba)

knn = KNeighborsClassifier(n_neighbors=5)
logistica = LogisticRegression(max_iter=2000, C=2, random_state=semilla)

knn.fit(caracteristicas_entreno, digito_entreno)
logistica.fit(caracteristicas_entreno, digito_entreno)

prediccion_knn = knn.predict(caracteristicas_prueba)
prediccion_logistica = logistica.predict(caracteristicas_prueba)

exactitud_knn = accuracy_score(digito_prueba, prediccion_knn)
exactitud_logistica = accuracy_score(digito_prueba, prediccion_logistica)

resultados_digitos = pd.DataFrame({
    'modelo': ['KNN', 'Regresión logística'],
    'accuracy': [exactitud_knn, exactitud_logistica]
})
resultados_digitos['accuracy'] = resultados_digitos['accuracy'].round(3)
resultados_digitos

In [ ]:
print('Reporte de clasificación para KNN')
print(classification_report(digito_prueba, prediccion_knn, zero_division=0))

figura, ejes = plt.subplots(1, 2, figsize=(14, 5))
clases = [str(numero) for numero in range(10)]

matriz_knn = confusion_matrix(digito_prueba, prediccion_knn, labels=clases)
matriz_logistica = confusion_matrix(digito_prueba, prediccion_logistica, labels=clases)

ConfusionMatrixDisplay(matriz_knn, display_labels=clases).plot(ax=ejes[0], colorbar=False, cmap='Blues')
ejes[0].set_title(f'KNN - accuracy {exactitud_knn:.3f}')
ConfusionMatrixDisplay(matriz_logistica, display_labels=clases).plot(ax=ejes[1], colorbar=False, cmap='Oranges')
ejes[1].set_title(f'Regresión logística - accuracy {exactitud_logistica:.3f}')
plt.tight_layout()
plt.show()

## 5. DTW en el par con más confusiones

Busco el error más frecuente de KNN y comparo un audio de cada dígito usando DTW. En vez de comparar muestras de amplitud, uso sus MFCC porque resumen mejor la forma en que se escucha cada palabra.

In [ ]:
matriz_sin_diagonal = matriz_knn.copy()
np.fill_diagonal(matriz_sin_diagonal, 0)
fila_error, columna_error = np.unravel_index(np.argmax(matriz_sin_diagonal), matriz_sin_diagonal.shape)
digito_real = clases[fila_error]
digito_predicho = clases[columna_error]
cantidad_error = matriz_sin_diagonal[fila_error, columna_error]

print(f'Error más repetido en KNN: {digito_real} se predijo como {digito_predicho} ({cantidad_error} veces)')

audio_a, sr_a = librosa.load(datos[datos['digito'] == digito_real].iloc[0]['archivo'], sr=frecuencia_muestreo)
audio_b, sr_b = librosa.load(datos[datos['digito'] == digito_predicho].iloc[0]['archivo'], sr=frecuencia_muestreo)
mfcc_a = librosa.feature.mfcc(y=audio_a, sr=sr_a, n_mfcc=13, hop_length=128)
mfcc_b = librosa.feature.mfcc(y=audio_b, sr=sr_b, n_mfcc=13, hop_length=128)

matriz_dtw, camino_dtw = librosa.sequence.dtw(X=mfcc_a, Y=mfcc_b, metric='euclidean')
distancia_dtw = matriz_dtw[-1, -1] / len(camino_dtw)
print(f'Distancia DTW promedio entre {digito_real} y {digito_predicho}: {distancia_dtw:.2f}')

plt.figure(figsize=(7, 5))
plt.imshow(matriz_dtw, origin='lower', aspect='auto', cmap='magma')
plt.plot(camino_dtw[:, 1], camino_dtw[:, 0], color='cyan', linewidth=1)
plt.colorbar(label='distancia acumulada')
plt.xlabel(f'frames MFCC del dígito {digito_predicho}')
plt.ylabel(f'frames MFCC del dígito {digito_real}')
plt.title('Alineación DTW del par más confundido')
plt.show()

## 6. Identificación del hablante

Finalmente uso las mismas características, pero ahora el objetivo es reconocer a la persona que hizo la grabación.

In [ ]:
caracteristicas_entreno_h, caracteristicas_prueba_h, hablante_entreno, hablante_prueba = train_test_split(
    matriz_caracteristicas, datos['hablante'], test_size=.20, random_state=semilla, stratify=datos['hablante']
)

escalador_hablantes = StandardScaler()
caracteristicas_entreno_h = escalador_hablantes.fit_transform(caracteristicas_entreno_h)
caracteristicas_prueba_h = escalador_hablantes.transform(caracteristicas_prueba_h)

modelo_hablantes = KNeighborsClassifier(n_neighbors=3)
modelo_hablantes.fit(caracteristicas_entreno_h, hablante_entreno)
prediccion_hablantes = modelo_hablantes.predict(caracteristicas_prueba_h)
exactitud_hablantes = accuracy_score(hablante_prueba, prediccion_hablantes)

print(f'Accuracy para identificar al hablante: {exactitud_hablantes:.3f}')
print(classification_report(hablante_prueba, prediccion_hablantes, zero_division=0))

matriz_hablantes = confusion_matrix(hablante_prueba, prediccion_hablantes, labels=sorted(datos['hablante'].unique()))
plt.figure(figsize=(7, 6))
ConfusionMatrixDisplay(matriz_hablantes, display_labels=sorted(datos['hablante'].unique())).plot(cmap='Greens', colorbar=False)
plt.title('Matriz de confusión: identificación de hablante')
plt.show()

## Conclusiones

- Los espectrogramas Mel muestran que los dígitos no solo cambian de duración: también tienen distinta distribución de energía en frecuencia.
- Las características MFCC y espectrales permiten separar varios dígitos sin usar una red neuronal.
- La matriz de confusión ayuda a ver cuáles sonidos se parecen más para el modelo. DTW permite revisar esa similitud aun cuando los audios no duran exactamente lo mismo.
- Reconocer al hablante puede resultar más fácil que reconocer el dígito, ya que la voz deja patrones propios en la señal.